In [8]:
"""
Filter Cityscapes für HPC

Dieses Skript kopiert nur die relevanten Dateien aus dem Cityscapes-Datensatz:
- _leftImg8bit.png (Bilder)
- _gtFine_labelTrainIds.png (Label für Training)

Die Dateien _gtFine_color.png und _gtFine_labelIds.png werden nicht kopiert.
"""

import shutil
from pathlib import Path
from tqdm import tqdm


def copy_cityscapes_filtered(source_dir, target_dir, annotation_type="gtFine"):
    """
    Kopiert nur _leftImg8bit.png und _labelTrainIds.png aus dem Cityscapes-Datensatz.
    
    Args:
        source_dir: Pfad zum originalen Cityscapes-Datensatz
        target_dir: Zielpfad für den gefilterten Datensatz
        annotation_type: "gtFine" oder "gtCoarse"
    """
    source_dir = Path(source_dir)
    target_dir = Path(target_dir)
    
    if annotation_type == "gtFine":
        images_dir = "leftImg8bit_trainvaltest/leftImg8bit"
        gt_dir = "gtFine_trainvaltest/gtFine"
        print("Filtering gtFine dataset...")
    elif annotation_type == "gtCoarse":
        images_dir = "leftImg8bit_trainextra/leftImg8bit"
        gt_dir = "gtCoarse/gtCoarse"
        print("Filtering gtCoarse dataset...")
    else:
        raise ValueError('Invalid annotation type. Use "gtFine" or "gtCoarse".')
    
    
    source_images = source_dir / images_dir
    source_labels = source_dir / gt_dir
    print(f"Source Images: {source_images}")
    print(f"Source Labels: {source_labels}")
    
    # Alle Splits durchgehen (train, val, test)
    splits = [d for d in source_images.iterdir() if d.is_dir()]
    print(f"Found splits: {[split.name for split in splits]}")
    
    total_files = 0
    for split_dir in splits:
        cities = [c for c in split_dir.iterdir() if c.is_dir()]
        for city_dir in cities:
            total_files += len(list(city_dir.glob("*_leftImg8bit.png")))
    
    with tqdm(total=total_files, desc=f"Copying {annotation_type}", unit="file") as pbar:
        for split_dir in splits:
            split = split_dir.name
            cities = [c for c in split_dir.iterdir() if c.is_dir()]
            print(f"Found cities: {[c.name for c in cities]}")

            for city_dir in cities:
                city = city_dir.name
                
                # Alle Images in diesem City-Ordner finden
                image_files = sorted(city_dir.glob("*_leftImg8bit.png"))
                
                for image_file in image_files:
                    # Basis-ID aus Dateiname extrahieren
                    image_id = image_file.stem.replace("_leftImg8bit", "")
                    
                    # Entsprechende Label-Datei finden
                    label_file = source_labels / split / city / f"{image_id}_{annotation_type}_labelTrainIds.png"
                    
                    # Zielverzeichnisse erstellen
                    target_image_dir = target_dir / images_dir / split / city
                    target_label_dir = target_dir / gt_dir / split / city
                    
                    target_image_dir.mkdir(parents=True, exist_ok=True)
                    target_label_dir.mkdir(parents=True, exist_ok=True)
                    
                    # Dateien kopieren
                    if image_file.exists():
                        shutil.copy(image_file, target_image_dir)
                    else:
                        print(f"WARNING: Image not found: {image_file}")
                    
                    if label_file.exists():
                        shutil.copy(label_file, target_label_dir)
                    else:
                        print(f"WARNING: Label not found: {label_file}")
                    
                    pbar.update(1)

In [9]:
source_dir = "../data/Cityscapes"
target_dir = "../data/cityscapes_hpc"
copy_cityscapes_filtered(source_dir, target_dir, annotation_type="gtFine")

Filtering gtFine dataset...
Source Images: ..\data\Cityscapes\leftImg8bit_trainvaltest\leftImg8bit
Source Labels: ..\data\Cityscapes\gtFine_trainvaltest\gtFine
Found splits: ['test', 'train', 'val']


Copying gtFine:   0%|          | 21/5000 [00:00<00:27, 183.45file/s]

Found cities: ['berlin', 'bielefeld', 'bonn', 'leverkusen', 'mainz', 'munich']


Copying gtFine:  31%|███       | 1545/5000 [00:08<00:21, 158.21file/s]

Found cities: ['aachen', 'bochum', 'bremen', 'cologne', 'darmstadt', 'dusseldorf', 'erfurt', 'hamburg', 'hanover', 'jena', 'krefeld', 'monchengladbach', 'strasbourg', 'stuttgart', 'tubingen', 'ulm', 'weimar', 'zurich']


Copying gtFine:  91%|█████████ | 4532/5000 [00:24<00:02, 197.49file/s]

Found cities: ['frankfurt', 'lindau', 'munster']


Copying gtFine: 100%|██████████| 5000/5000 [00:26<00:00, 188.15file/s]


In [10]:
copy_cityscapes_filtered(source_dir, target_dir, annotation_type="gtCoarse")

Filtering gtCoarse dataset...
Source Images: ..\data\Cityscapes\leftImg8bit_trainextra\leftImg8bit
Source Labels: ..\data\Cityscapes\gtCoarse\gtCoarse
Found splits: ['train_extra']


Copying gtCoarse:   0%|          | 5/19998 [00:00<07:20, 45.38file/s]

Found cities: ['augsburg', 'bad-honnef', 'bamberg', 'bayreuth', 'dortmund', 'dresden', 'duisburg', 'erlangen', 'freiburg', 'heidelberg', 'heilbronn', 'karlsruhe', 'konigswinter', 'konstanz', 'mannheim', 'muhlheim-ruhr', 'nuremberg', 'oberhausen', 'saarbrucken', 'schweinfurt', 'troisdorf', 'wuppertal', 'wurzburg']


Copying gtCoarse: 100%|██████████| 19998/19998 [05:24<00:00, 61.71file/s]
